# ELN Submission API

Use these endpoints to submit data from Electronic Lab Notebook systems into nmrXiv and check processing status.

The dev API docs currently support `chemotion` and `nobs` as ELN identifiers.

## Endpoints covered

- `POST /api/v1/{eln}/upload` - upload and process data from an ELN system.
- `GET /api/v1/{eln}/status/{external_id}` - get processing status for an ELN submission.

Both endpoints require authentication with a Bearer token.

In [ ]:
import os
from pprint import pprint

import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("NMRXIV_BASE_URL", "https://dev.nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api/v1"
AUTH_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

BASE_URL, API_BASE

In [ ]:
def api_request(method, url, **kwargs):
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def extract_token(payload):
    if not isinstance(payload, dict):
        return None
    token = payload.get("access_token") or payload.get("token")
    if token:
        return token
    data = payload.get("data")
    if isinstance(data, dict):
        return data.get("access_token") or data.get("token")
    return None


def require_bearer_token():
    if "Authorization" not in session.headers:
        raise RuntimeError("Login first so the session has an Authorization: Bearer <token> header.")


def eln_upload(eln, external_id, callback_url, zip_url, release_date=None):
    require_bearer_token()
    payload = {"external_id": external_id, "callback_url": callback_url, "zip_url": zip_url}
    if release_date:
        payload["release_date"] = release_date
    return api_request("POST", f"{API_BASE}/{eln}/upload", json=payload)


def eln_status(eln, external_id):
    require_bearer_token()
    return api_request("GET", f"{API_BASE}/{eln}/status/{external_id}")

## Login

Set `NMRXIV_EMAIL` and `NMRXIV_PASSWORD` in `.env`. The returned token is stored only in this notebook session.

In [ ]:
email = os.getenv("NMRXIV_EMAIL")
password = os.getenv("NMRXIV_PASSWORD")

if email and password:
    login_response = api_request("POST", f"{AUTH_BASE}/auth/login", json={"email": email, "password": password})
    token = extract_token(login_response)
    if token:
        session.headers.update({"Authorization": f"Bearer {token}"})
        print("Bearer token configured for this notebook session.")
    else:
        print("Login succeeded, but no token key was recognized. Inspect login_response if needed.")
else:
    print("Set NMRXIV_EMAIL and NMRXIV_PASSWORD in .env to run authenticated ELN examples.")

## Upload and process data from Electronic Lab Notebook (ELN) systems​

Creates or updates a draft with data from external ELN systems. Currently supports Chemotion and NoBs. Processes ZIP files containing experimental data and extracts them to organized folder structure.. Required fields are:

- `eln`: `chemotion` or `nobs`
- `external_id`: unique identifier from the ELN system
- `callback_url`: URL nmrXiv should call after processing
- `zip_url`: URL of a ZIP file containing experimental data
- `release_date`: optional ISO date, for example `2026-12-31`

This cell is guarded by `NMRXIV_RUN_ELN_UPLOAD=false` by default so it does not accidentally create/update a draft.

In [ ]:
eln = os.getenv("NMRXIV_ELN", "chemotion")
external_id = "CHEM-EXP-2024-001"
callback_url = os.getenv("NMRXIV_ELN_CALLBACK_URL", "https://chemotion.example.com/api/callback")
zip_url = "https://chemotion.example.com/exports/experiment-data.zip"
release_date = "2026-12-31" or None

run_upload = os.getenv("NMRXIV_RUN_ELN_UPLOAD", "false").lower() == "true"

if run_upload:
    upload_response = eln_upload(eln, external_id, callback_url, zip_url, release_date)
    pprint(upload_response)
else:
    print("Set NMRXIV_RUN_ELN_UPLOAD=true in .env to submit ELN data.")
    pprint({
        "eln": eln,
        "external_id": external_id,
        "callback_url": callback_url,
        "zip_url": zip_url,
        "release_date": release_date,
    })

## Get processing status of ELN submission​

Retrieves the current processing status and details of a draft submission identified by external_id from the specified ELN system. Use the same `eln` and `external_id` used for upload. This endpoint also requires a Bearer token.

In [ ]:
run_status_check = os.getenv("NMRXIV_RUN_ELN_STATUS", "false").lower() == "true"

if run_status_check:
    status_response = eln_status(eln, external_id)
    pprint(status_response)
else:
    print("Set NMRXIV_RUN_ELN_STATUS=true in .env to check an ELN submission status.")
    pprint({"eln": eln, "external_id": external_id})